In [ ]:
import numpy as np
import numba as nb
import math
from time import perf_counter
import matplotlib.pyplot as plt

from gridcp import (
    OnlineChangepointDetector,
    make_univariate_mean_change_detector,
    make_univariate_variance_change_detector,
    make_univariate_mean_or_variance_change_detector,
    make_multivariate_mean_change_detector,
    make_multivariate_mean_or_covariance_change_detector,
)
from gridcp.utils import fastlog
from gridcp.calibration import draw_samples, mc_max_statistics
from gridcp import builtins

## Univariate change-in-mean

### Known variance

In [ ]:
N = 50
K_calibrate = 10000  # for threshold calibration
K = 1000  # for performance evaluation
p = 1

detector = make_univariate_mean_change_detector(penalty_constant=1.0)
detector.calibrate_false_alarm(
    alpha=0.05, N=N, K=K_calibrate, null_dist=np.random.normal
)
critical_value_known_var = detector._state["penalty_constant"]
print("critical value = ", critical_value_known_var)

In [ ]:
changepoint_loc = N // 2
prechange_dist = np.random.normal
postchange_dist = np.random.normal
prechange_kwargs = {"loc": 0, "scale": 1, "size": (p,)}
postchange_kwargs = {"loc": 2, "scale": 1, "size": (p,)}
xs = draw_samples(
    K=K,
    N=N,
    p=p,
    seed=123,
    pre_change_dist=prechange_dist,
    pre_change_kwargs=prechange_kwargs,
    post_change_dist=postchange_dist,
    post_change_kwargs=postchange_kwargs,
    changepoint_loc=changepoint_loc,
)

detect_times = np.ones(K, dtype=np.int64) * N
for k in range(K):
    detector = make_univariate_mean_change_detector(
        penalty_constant=critical_value_known_var
    )
    for i in range(N):
        x_new = xs[k, i]
        detector.update(x_new)
        if detector.alarm:
            detect_times[k] = i + 1
            break

print(f"Average detection delay: {np.mean(detect_times) - changepoint_loc}")

### Uknown variance

In [ ]:
detector = make_univariate_mean_change_detector(
    penalty_constant=1.0, mode="unknown_variance"
)
detector.calibrate_false_alarm(
    alpha=0.05, N=N, K=K_calibrate, null_dist=np.random.normal
)
critical_value_unknown_var = detector._state["penalty_constant"]
print("critical value = ", critical_value_unknown_var)

In [ ]:
changepoint_loc = N // 2
prechange_dist = np.random.normal
postchange_dist = np.random.normal
prechange_kwargs = {"loc": 0, "scale": 1, "size": (p,)}
postchange_kwargs = {"loc": 2, "scale": 1, "size": (p,)}
xs = draw_samples(
    K=K,
    N=N,
    p=p,
    seed=123,
    pre_change_dist=prechange_dist,
    pre_change_kwargs=prechange_kwargs,
    post_change_dist=postchange_dist,
    post_change_kwargs=postchange_kwargs,
    changepoint_loc=changepoint_loc,
)

detect_times = np.ones(K, dtype=np.int64) * N
for k in range(K):
    detector = make_univariate_mean_change_detector(
        penalty_constant=critical_value_unknown_var, mode="unknown_variance"
    )
    for i in range(N):
        x_new = xs[k, i]
        detector.update(x_new)
        if detector.alarm:
            detect_times[k] = i + 1
            break

print(f"Average detection delay: {np.mean(detect_times) - changepoint_loc}")

### Change in mean or variance

In [ ]:
detector = make_univariate_mean_or_variance_change_detector(penalty_constant=1.0)
detector.calibrate_false_alarm(
    alpha=0.05, N=N, K=K_calibrate, null_dist=np.random.normal
)
critical_value_meanvariance = detector._state["penalty_constant"]
print("critical value = ", critical_value_meanvariance)

In [ ]:
changepoint_loc = N // 2
prechange_dist = np.random.normal
postchange_dist = np.random.normal
prechange_kwargs = {"loc": 0, "scale": 1, "size": (p,)}
postchange_kwargs = {"loc": 2, "scale": 1, "size": (p,)}
xs = draw_samples(
    K=K,
    N=N,
    p=p,
    seed=123,
    pre_change_dist=prechange_dist,
    pre_change_kwargs=prechange_kwargs,
    post_change_dist=postchange_dist,
    post_change_kwargs=postchange_kwargs,
    changepoint_loc=changepoint_loc,
)

detect_times = np.ones(K, dtype=np.int64) * N
for k in range(K):
    detector = make_univariate_mean_or_variance_change_detector(
        penalty_constant=critical_value_meanvariance
    )
    for i in range(N):
        x_new = xs[k, i]
        detector.update(x_new)
        if detector.alarm:
            detect_times[k] = i + 1
            break

print(f"Average detection time: {np.mean(detect_times) - changepoint_loc}")

In [ ]:
K = 1000
mus = np.linspace(0, 3, 10)


changepoint_loc = N // 2
prechange_dist = np.random.normal
postchange_dist = np.random.normal

dd1 = np.ones((len(mus), K)) * N
dd2 = np.ones((len(mus), K)) * N
dd3 = np.ones((len(mus), K)) * N


for j in range(len(mus)):
    prechange_kwargs = {"loc": 0, "scale": 1, "size": (p,)}
    postchange_kwargs = {"loc": mus[j], "scale": 1, "size": (p,)}
    xs = draw_samples(
        K=K,
        N=N,
        p=p,
        seed=1234,
        pre_change_dist=prechange_dist,
        pre_change_kwargs=prechange_kwargs,
        post_change_dist=postchange_dist,
        post_change_kwargs=postchange_kwargs,
        changepoint_loc=changepoint_loc,
    )

    for k in range(K):
        detector1 = make_univariate_mean_change_detector(
            penalty_constant=critical_value_known_var
        )
        detector2 = make_univariate_mean_change_detector(
            penalty_constant=critical_value_unknown_var,
            mode="unknown_variance",
        )
        detector3 = make_univariate_mean_or_variance_change_detector(
            penalty_constant=critical_value_meanvariance
        )
        for i in range(N):
            x_new = xs[k, i]
            detector1.update(x_new)
            if detector1.alarm:
                dd1[j, k] = i + 1
                break
        for i in range(N):
            x_new = xs[k, i]
            detector2.update(x_new)
            if detector2.alarm:
                dd2[j, k] = i + 1
                break
        for i in range(N):
            x_new = xs[k, i]
            detector3.update(x_new)
            if detector3.alarm:
                dd3[j, k] = i + 1
                break

In [ ]:
print("FA 1:", np.mean(dd1[0] < N))
print("FA 2:", np.mean(dd2[0] < N))
print("FA 3:", np.mean(dd3[0] < N))

In [ ]:
avg_dd1 = np.mean(dd1, axis=1) - changepoint_loc
avg_dd2 = np.mean(dd2, axis=1) - changepoint_loc
avg_dd3 = np.mean(dd3, axis=1) - changepoint_loc

plt.plot(mus, avg_dd1, label="mean change detector")
plt.plot(mus, avg_dd2, label="mean change detector unknown variance")
plt.plot(mus, avg_dd3, label="mean and variance change detector")
plt.xlabel("mean change magnitude")
plt.ylabel("average detection delay")
plt.legend()
plt.show()

## Univariate variance change

### Known mean

In [ ]:
K = 1000

detector = make_univariate_variance_change_detector(penalty_constant=1.0)
detector.calibrate_false_alarm(
    alpha=0.05, N=N, K=K_calibrate, null_dist=np.random.normal
)
critical_value_known_mean = detector._state["penalty_constant"]

In [ ]:
changepoint_loc = N // 2
prechange_dist = np.random.normal
postchange_dist = np.random.normal
prechange_kwargs = {"loc": 0, "scale": 1, "size": (p,)}
postchange_kwargs = {"loc": 0, "scale": 4, "size": (p,)}
xs = draw_samples(
    K=K,
    N=N,
    p=p,
    seed=123,
    pre_change_dist=prechange_dist,
    pre_change_kwargs=prechange_kwargs,
    post_change_dist=postchange_dist,
    post_change_kwargs=postchange_kwargs,
    changepoint_loc=changepoint_loc,
)

detect_times = np.ones(K, dtype=np.int64) * N
for k in range(K):
    detector = make_univariate_variance_change_detector(
        penalty_constant=critical_value_known_mean
    )
    for i in range(N):
        x_new = xs[k, i]
        detector.update(x_new)
        if detector.alarm:
            detect_times[k] = i + 1
            break

print(f"Average detection time: {np.mean(detect_times) - changepoint_loc}")

In [ ]:
K = 1000
postchangesds = np.linspace(1, 5, 10)


changepoint_loc = N // 2
prechange_dist = np.random.normal
postchange_dist = np.random.normal

dd1 = np.ones((len(postchangesds), K)) * N
dd2 = np.ones((len(postchangesds), K)) * N


for j in range(len(postchangesds)):
    prechange_kwargs = {"loc": 0, "scale": 1, "size": (p,)}
    postchange_kwargs = {"loc": 0, "scale": postchangesds[j], "size": (p,)}
    xs = draw_samples(
        K=K,
        N=N,
        p=p,
        seed=1234,
        pre_change_dist=prechange_dist,
        pre_change_kwargs=prechange_kwargs,
        post_change_dist=postchange_dist,
        post_change_kwargs=postchange_kwargs,
        changepoint_loc=changepoint_loc,
    )

    for k in range(K):
        detector1 = make_univariate_variance_change_detector(
            penalty_constant=critical_value_known_mean
        )
        detector2 = make_univariate_mean_or_variance_change_detector(
            penalty_constant=critical_value_meanvariance
        )
        for i in range(N):
            x_new = xs[k, i]
            detector1.update(x_new)
            if detector1.alarm:
                dd1[j, k] = i + 1
                break
        for i in range(N):
            x_new = xs[k, i]
            detector2.update(x_new)
            if detector2.alarm:
                dd2[j, k] = i + 1
                break

In [ ]:
print("FA 1:", np.mean(dd1[0] < N))
print("FA 2:", np.mean(dd2[0] < N))

In [ ]:
avg_dd1 = np.mean(dd1, axis=1) - changepoint_loc
avg_dd2 = np.mean(dd2, axis=1) - changepoint_loc

plt.plot(postchangesds, avg_dd1, label="mean change detector")
plt.plot(postchangesds, avg_dd2, label="mean and variance change detector")
plt.xlabel("post-change standard deviation")
plt.ylabel("average detection delay")
plt.legend()
plt.show()

### Multivariate Gaussian CUSUM example (dense change)

In [ ]:
@nb.njit
def h(y):
    return y


@nb.njit
def f(sum_pre_j, sum_post_j, g, t):
    res = math.sqrt(1.0 * g / (t * (t - g))) * sum_pre_j
    res = res - math.sqrt(1.0 * (t - g) / t / g) * sum_post_j
    return np.sum(res * res)


@nb.njit
def penalty(g, t, p):
    logg = fastlog(t / 0.05)
    rr = math.sqrt(p * logg) + logg

    return rr

In [ ]:
N = 1000
K = 1000
p = 10
null_dist = np.random.normal
null_args = (0, 1)  # mean=0, std=1
sample = mc_max_statistics(
    N=N,
    K=K,
    p=p,
    h=h,
    f=f,
    penalty=penalty,
    null_dist=null_dist,
    penalty_constant=0.0,
    null_args=null_args,
    null_kwargs=None,
    auxiliary_data=None,
)

In [ ]:
critical_value = np.percentile(sample, 95)
print(f"Critical value at 95%: {critical_value}")

In [ ]:
xs = np.random.normal(0, 1, (N, p))
xs[(N // 2) :] += 0.2  # Introduce a change point at iteration 8000

state = init_state(p, h, f, penalty, critical_value)
for i in range(N):
    x_new = xs[i]
    update(x_new, state)
    if state["alarm"]:
        print(
            f"Alarm triggered at iteration {i+1} with maxx = {state['maxx']} at position {state['maxpos']}"
        )
        break

### Multivariate change in mean or covariance

In [ ]:
@nb.njit
def h_mean_cov(y):
    """
    Sufficient statistic for Gaussian mean+covariance:
    Return concatenation of y and vec(y y^T).

    If y has shape (p,), this returns an array of shape (p + p*p,)
    where the last p*p entries are column-major flattening of y y^T.
    """
    p = y.shape[0]
    yy = np.outer(y, y)
    out = np.empty((p + 1, p), dtype=y.dtype)
    out[0] = y
    out[1:] = yy

    return out


@nb.njit
def f_mean_cov(sum_pre_j, sum_post_j, g, t):
    """
    GLR-type statistic for a change in both mean and covariance
    in multivariate Gaussian data.

    sum_pre_j: sum of h(y) over segment 1  (shape (p+1, p))
    sum_post_j: sum of h(y) over segment 2 (shape (p+1, p))
    g: candidate change-point (segment 1 length)
    t: total sample size
    """
    n1 = t - g
    n2 = g

    p = sum_pre_j.shape[0]  # dimension of data
    if n1 <= p or n2 <= p:
        return 0.0

    totalsum = sum_pre_j + sum_post_j
    sum_pre_j_id = sum_pre_j[0]
    sum_post_j_id = sum_post_j[0]
    totalsum_id = totalsum[0]

    sum_pre_j_cov = sum_pre_j[1:]
    sum_post_j_cov = sum_post_j[1:]
    totalsum_cov = totalsum[1:]

    Sigma_tot = (totalsum_cov - np.outer(totalsum_id, totalsum_id) / t) / t
    Sigma_pre_j = (sum_pre_j_cov - np.outer(sum_pre_j_id, sum_pre_j_id) / n1) / n1
    Sigma_post_j = (sum_post_j_cov - np.outer(sum_post_j_id, sum_post_j_id) / n2) / n2

    # GLR statistic: t * log|Sigma| - g* log|Sigma1| - (t-g) * log|Sigma2|
    # Use slogdet for numerical stability
    sign0, logdet0 = np.linalg.slogdet(Sigma_tot)
    sign1, logdet1 = np.linalg.slogdet(Sigma_pre_j)
    sign2, logdet2 = np.linalg.slogdet(Sigma_post_j)

    if False:
        # If any covariance is singular (sign <= 0), treat statistic as 0
        if sign0 <= 0 or sign1 <= 0 or sign2 <= 0:
            if sign0 <= 0:
                print("Sigma_tot is singular or not positive definite")
            if sign1 <= 0:
                print("Sigma_pre_j is singular or not positive definite")
            if sign2 <= 0:
                print("Sigma_post_j is singular or not positive definite")
            return 0.0

    LR = t * logdet0 - n1 * logdet1 - n2 * logdet2
    df = p + (p * (p + 1)) // 2  # Number of parameters in mean+covariance

    return LR - df


@nb.njit
def penalty_mean_cov(g, t, p):
    df = (p * (p + 1)) // 2 + p
    logg = fastlog(t / 0.05)
    rr = math.sqrt(df * logg) + logg

    return rr

In [ ]:
p = 5
S1 = np.zeros((p + 1, p))
S2 = np.zeros((p + 1, p))

T1 = np.zeros((p + 1, p))
T2 = np.zeros((p + 1, p))

NN = 100
for i in range(NN):
    y = np.random.normal(0, 1, p)
    h_y = h_mean_cov(y)
    S1 += h_y
    hy = h_mean_cov(y / 2)
    T1 += hy

    y = np.random.normal(0, 1, p)
    h_y = h_mean_cov(y)
    S2 += h_y
    hy = h_mean_cov(y / 2)
    T2 += hy


r1 = f_mean_cov(S1, S2, 50, 100)
r2 = f_mean_cov(T1, T2, 50, 100)
print(r1)
print(r2)

In [ ]:
N = 1000
K = 1000
p = 10
null_dist = np.random.normal
null_args = (0, 1)  # mean=0, std=1
sample = mc_max_statistics(
    N=N,
    K=K,
    p=p,
    h=h_mean_cov,
    f=f_mean_cov,
    penalty=penalty_mean_cov,
    null_dist=null_dist,
    penalty_constant=0.0,
    null_args=null_args,
    null_kwargs=None,
    auxiliary_data=None,
)

In [ ]:
critical_value = np.percentile(sample, 95)
print(f"Critical value at 95%: {critical_value}")

In [ ]:
xs = np.random.normal(0, 1, (N, p))
xs[(N // 2) :] += 0.4

state = init_state(p, h_mean_cov, f_mean_cov, penalty_mean_cov, critical_value)
for i in range(N):
    x_new = xs[i]
    update(x_new, state)
    if state["alarm"]:
        print(
            f"Alarm triggered at iteration {i+1} with maxx = {state['maxx']} at position {state['maxpos']}"
        )
        break

In [ ]:
xs = np.zeros((N, p))
xs[: (N // 2)] = np.random.normal(0, 1, (N // 2, p))
xs[(N // 2) :] = np.random.normal(0, 0.5, (N // 2, p))  # Introduce a changepoint


state = init_state(p, h_mean_cov, f_mean_cov, penalty_mean_cov, critical_value)
for i in range(N):
    x_new = xs[i]
    update(x_new, state)
    if state["alarm"]:
        print(
            f"Alarm triggered at iteration {i+1} with maxx = {state['maxx']} at position {state['maxpos']}"
        )
        break

In [ ]:
state

### General likelihood ratio test using Numba, requiring gradients

In [ ]:
## Specific examples are for Bernoulli with natural param theta = log(p/(1-p)) and A(theta) = log(1 + exp(theta))


from matplotlib.pylab import inf


@nb.njit
def h(y):
    return y


@nb.njit
def A_func(theta):
    return np.log(1.0 + np.exp(theta)).sum()


@nb.njit
def grad_A(theta):
    ee = np.exp(theta)
    return ee / (1.0 + ee)


@nb.njit
def hess_A(theta):
    H = np.zeros((1, 1), dtype=np.float64)
    ee = np.exp(theta)
    H[0, 0] = (ee / ((1.0 + ee) ** 2)).sum()
    return H


@nb.njit
def penalty(g, t, p):
    logg = fastlog(t / 0.05)
    rr = math.sqrt(p * logg) + logg

    return rr


## The below should be general


@nb.njit
def logLik(theta, S, n):
    # S is cumulative sum!
    # n is total sample size

    # if S is numerically zero or n, we return zero:
    if np.all(np.abs(S) < 1e-10) or np.all(np.abs(S - n) < 1e-10):
        return 0.0
    A_theta = A_func(theta)
    return np.dot(theta, S) - n * A_theta


@nb.njit
def newton_mle(
    S, n, theta_init, A_func, grad_A, hess_A, logLik, tol=1e-12, max_iter=1000
):
    """
    Maximize logLik(theta; S, n) = theta @ S - n * A_func(theta)
    over theta ∈ R^v using safeguarded Newton.

    Inputs
    ------
    S : 1D array (v,)
        Sum of sufficient statistics.
    n : int
        Number of observations.
    theta_init : 1D array (v,)
        Starting point (e.g. previous MLE or some fixed value).
    tol : float
        Stop when ||grad||_inf < tol.
    max_iter : int
        Hard cap on number of iterations.

    Returns
    -------
    theta_hat : 1D array (v,)
    converged : boolean
    """

    if (np.abs(S) < 1e-10).all():
        return np.array([-np.inf]), True

    if (np.abs(S - n) < 1e-10).all():
        return np.array([np.inf]), True

    theta = theta_init
    v = theta.shape[0]

    # Small ridge for numerical stability of Hessian
    ridge = 1e-8

    # current log-likelihood
    ll_old = logLik(theta, S, n)

    for it in range(max_iter):
        # Gradient of log-likelihood: S - n * grad_A(theta)
        gA = grad_A(theta)  # shape (v,)
        grad = S - n * gA  # shape (v,)

        # Check convergence: ||grad||_inf < tol
        max_abs_grad = 0.0
        for i in range(v):
            val = grad[i]
            if val < 0.0:
                val = -val
            if val > max_abs_grad:
                max_abs_grad = val
        if max_abs_grad < tol:
            return theta, True

        # Hessian of log-likelihood: H = -n * hess_A(theta)
        HA = hess_A(theta)  # shape (v, v)
        H = np.empty_like(HA)
        for i in range(v):
            for j in range(v):
                H[i, j] = -n * HA[i, j]
        # Add a small ridge to diagonal: H ← H − ridge * I
        # (H is negative semidefinite; we move it slightly more negative
        # to make it better-conditioned)
        for i in range(v):
            H[i, i] = H[i, i] - ridge

        # Solve H * step = grad
        # (instead of explicitly inverting H)
        try:
            step = np.linalg.solve(H, grad)
        except Exception:
            # If solve fails for some reason, abort
            return theta, False

        # Safeguard: start with full Newton step, shrink if ll decreases
        step_scale = 1.0
        theta_new = theta.copy()
        ll_new = ll_old

        # At most a few shrink steps (hard-coded; no user tuning)
        for _ in range(6):
            for i in range(v):
                theta_new[i] = theta[i] + step_scale * step[i]

            ll_candidate = logLik(theta_new, S, n)

            if np.isfinite(ll_candidate) and ll_candidate >= ll_old:
                ll_new = ll_candidate
                break  # accept this step
            else:
                step_scale *= 0.5  # shrink and try again

        # Update
        theta = theta_new
        ll_old = ll_new

    # If we fall out of the loop, we hit max_iter
    return theta, False


@nb.njit
def f(sum_pre_j, sum_post_j, g, t):
    print("###################")
    print("Iter ", t, " g ", g)
    theta_init = np.zeros(sum_pre_j.shape[0], dtype=np.float64)
    theta0, converged = newton_mle(
        sum_pre_j + sum_post_j, t, theta_init, A_func, grad_A, hess_A, logLik
    )
    print("theta0 ", theta0)
    print("converged = ", converged)
    theta1, converged1 = newton_mle(
        sum_pre_j, t - g, theta0, A_func, grad_A, hess_A, logLik
    )
    theta2, converged2 = newton_mle(
        sum_post_j, g, theta0, A_func, grad_A, hess_A, logLik
    )
    print("theta1 ", theta1)
    print("converged1 = ", converged1)
    print("theta2 ", theta2)
    print("converged2 = ", converged2)
    ll0 = logLik(theta0, sum_pre_j + sum_post_j, t)
    ll1 = logLik(theta1, sum_pre_j, t - g)
    ll2 = logLik(theta2, sum_post_j, g)
    print("ll0 = ", ll0)
    print("ll1 = ", ll1)
    print("ll2 = ", ll2)
    print("LR = ", ll1 + ll2 - ll0)
    return ll1 + ll2 - ll0

In [ ]:
p = 1
penalty_constant = 3.0
N = 10
xs = np.zeros(N)
p0 = 0.1
p1 = 0.8
xs[: (N // 2)] = np.random.binomial(n=1, p=p0, size=N // 2)
xs[(N // 2) :] = np.random.binomial(n=1, p=p1, size=N // 2)

xs[:10]

In [ ]:
state = init_state(p, h, f, penalty, penalty_constant)
for i in range(N):
    x_new = xs[i]
    update(x_new, state)
    if state["alarm"]:
        print(
            f"Alarm triggered at iteration {i+1} with maxx = {state['maxx']} at position {state['maxpos']}"
        )
        break

### General likelihood ratio test (non-Numba) using autograd. Fully automatic. sucks

In [ ]:
import autograd.numpy as anp
from autograd import grad
from scipy.optimize import minimize

## Currently, the binomial:


def h(y):
    return y


def A_func(theta):
    ## theta is natural parameter, for bernoulli, A(theta) = log(1 + exp(theta))
    ## and theta = log(p/(1-p)) where p is the Bernoulli paramete
    ret = anp.log(1.0 + anp.exp(theta))
    return ret


def logLik(theta, S, n):
    # S is cumulative sum!
    A_theta = A_func(theta)
    return anp.dot(theta, S) - n * A_theta


theta_init = anp.zeros(1)
df = 1
p = 1


def f(sum_pre_j, sum_post_j, g, t):
    # compute MLEs
    # Overall MLE:
    def objective(theta_flat):
        return -logLik(
            theta_flat, sum_pre_j + sum_post_j, t
        )  # negative for minimization

    def objective_grad(theta_flat):
        theta = theta_flat
        gg = grad(logLik, argnum=0)(theta, sum_pre_j + sum_post_j, t)  # shape (v,)
        return -gg  # because we minimize the negative

    res = minimize(
        objective,
        x0=theta_init,
        jac=objective_grad,
        method="L-BFGS-B",
        options={"maxiter": 200, "disp": False},
    )
    theta0 = res.x

    def objective(theta_flat):
        return -logLik(theta_flat, sum_pre_j, t - g)  # negative for minimization

    def objective_grad(theta_flat):
        theta = theta_flat
        gg = grad(logLik, argnum=0)(theta, sum_pre_j, t - g)  # shape (v,)
        return -gg  # because we minimize the negative

    res = minimize(
        objective,
        x0=theta0,
        jac=objective_grad,
        method="L-BFGS-B",
        options={"maxiter": 200, "disp": False},
    )
    theta1 = res.x

    def objective(theta_flat):
        return -logLik(theta_flat, sum_post_j, g)  # negative for minimization

    def objective_grad(theta_flat):
        theta = theta_flat
        gg = grad(logLik, argnum=0)(theta, sum_post_j, g)  # shape (v,)
        return -gg  # because we minimize the negative

    res = minimize(
        objective,
        x0=theta0,
        jac=objective_grad,
        method="L-BFGS-B",
        options={"maxiter": 200, "disp": False},
    )
    theta2 = res.x

    ret = (
        logLik(theta0, sum_pre_j + sum_post_j, t)
        + logLik(theta1, sum_pre_j, t - g)
        + logLik(theta2, sum_post_j, g)
        - df
    )
    print("ret = ", ret)
    return ret


@nb.njit
def penalty(g, t, p):
    logg = fastlog(t / 0.05)
    rr = math.sqrt(p * logg) + logg

    return rr

In [ ]:
p = 1
penalty_constant = 3.0
N = 100
xs = np.zeros(N)
p0 = 0.5
p1 = 0.8
xs[: (N // 2)] = np.random.binomial(n=1, p=p0, size=N // 2)
xs[(N // 2) :] = np.random.binomial(n=1, p=p1, size=N // 2)

state = init_state(p, h, f, penalty, penalty_constant)
for i in range(N):
    x_new = xs[i]
    update(x_new, state)
    if state["alarm"]:
        print(
            f"Alarm triggered at iteration {i+1} with maxx = {state['maxx']} at position {state['maxpos']}"
        )
        break

In [ ]:
state